# Session 3 lab · Image processing & filtering

The code behind the slides. Each cell shows **one idea** in a few lines:

1. run it and look at the result;
2. change a value marked **`· try`** (the values after it are good ones to try) and run it again;
3. read *What to notice* under the cell.

| Part | Ideas |
|---|---|
| **A · Convolution** | point operations, convolution by hand, the impulse response, padding, output size |
| **B · Smoothing & sharpening** | box vs Gaussian, noise ÷ √n, separable filters, unsharp masking |
| **C · Edges** | a brightness profile, Sobel, gradient strength and direction, Canny |
| **D · Frequency & sampling** | a sum of waves, the Fourier transform, aliasing, blur before you shrink, pyramids |

Nothing to hand in, nothing to upload — it runs on free Colab.

In [ ]:
# Setup — run this cell first
import numpy as np
import matplotlib.pyplot as plt
import cv2, time
from skimage import data

photo = data.astronaut()                          # a 512 × 512 colour photo (NASA, public domain)
gray = cv2.cvtColor(photo, cv2.COLOR_RGB2GRAY)    # the same photo in grey, 0–255
rng = np.random.default_rng(0)                    # random numbers for the noise cells
print('ready')

## A · Convolution

A **point operation** changes each pixel on its own. A **convolution** slides a small grid of weights — the **kernel** — over the image: at each place, multiply and add.

In [ ]:
# A1 · Point operations: out = T(in), the same rule for every pixel
operation = 'gamma'   # 'brightness', 'contrast', 'invert' or 'gamma' · try 'contrast', 'invert'
amount = 0.5          # brightness: + amount×100 · contrast: × (1 + amount) · gamma: power · try 2.0

x = gray.astype(float)
if operation == 'brightness': out = x + amount * 100
if operation == 'contrast':   out = 128 + (1 + amount) * (x - 128)
if operation == 'invert':     out = 255 - x
if operation == 'gamma':      out = 255 * (x / 255) ** amount
out = np.clip(out, 0, 255)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
ax[0].imshow(gray, cmap='gray', vmin=0, vmax=255); ax[0].set_title('in')
ax[1].imshow(out, cmap='gray', vmin=0, vmax=255);  ax[1].set_title(f'out · {operation}')
ax[2].plot(x.ravel()[::97], out.ravel()[::97], '.', ms=2); ax[2].set_xlabel('in'); ax[2].set_ylabel('out'); ax[2].set_title('the mapping')
ax[0].axis('off'); ax[1].axis('off'); plt.tight_layout(); plt.show()

**What to notice:** the right-hand plot is a single curve — every pixel follows the same rule, whatever its neighbours. That is why a point operation cannot remove noise.

In [ ]:
# A2 · Convolution by hand: slide a 3 × 3 kernel, multiply, add (the stepper on the slides)
img = np.array([[22, 25, 21, 198, 205, 201, 203], [24, 23, 26, 202, 199, 204, 200], [21, 26, 24, 200, 203, 197, 202],
                [25, 22, 250, 204, 198, 202, 199], [23, 24, 22, 199, 201, 203, 205], [26, 21, 25, 203, 200, 198, 201],
                [24, 25, 23, 201, 204, 202, 200]], float)
kernel_name = 'average'   # 'average', 'sharpen' or 'edge' · try 'sharpen', 'edge'

kernels = {'average': np.ones((3, 3)) / 9,
           'sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
           'edge':    np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], float)}
k = kernels[kernel_name]
out = np.zeros((5, 5))                   # 'valid': only where the 3 × 3 window fits
for i in range(5):
    for j in range(5):
        window = img[i:i + 3, j:j + 3]
        out[i, j] = (window * k).sum()
print(np.round(out).astype(int))
print('same as OpenCV:', np.allclose(out, cv2.filter2D(img, -1, k)[1:-1, 1:-1]))

**What to notice:** with `'average'`, row 3 reads 49 108 167 201 201 — as on the slides: the lone 250 is diluted by its eight neighbours, and the sharp edge becomes a ramp. Two loops and one multiply-add are all a convolution is.

In [ ]:
# A3 · A single bright dot shows the kernel (impulse response)
kernel = np.arange(1, 10, dtype=float).reshape(3, 3)    # 1 … 9, not symmetric
mode = 'correlation'       # 'correlation' (what OpenCV does) or 'convolution' · try 'convolution'

dot = np.zeros((7, 7)); dot[3, 3] = 1
if mode == 'correlation':
    out = cv2.filter2D(dot, -1, kernel)                  # lays the kernel on as it is
else:
    out = cv2.filter2D(dot, -1, cv2.flip(kernel, -1))    # rotate the kernel 180° first
print('kernel:\n', kernel.astype(int))
print(f'{mode} of a single dot:\n', out[2:5, 2:5].astype(int))

**What to notice:** convolution copies the kernel; correlation copies it rotated 180°. For a symmetric kernel (like the average) both give the same answer — and for kernels a CNN learns (session 4) the difference does not matter.

In [ ]:
# A4 · Padding: what is outside the image?
border = 'constant'   # 'constant' (zeros), 'replicate' or 'reflect' · try 'replicate', 'reflect'
size = 41             # a big box blur makes the border visible · try 15

modes = {'constant': cv2.BORDER_CONSTANT, 'replicate': cv2.BORDER_REPLICATE, 'reflect': cv2.BORDER_REFLECT_101}
blurred = cv2.blur(photo, (size, size), borderType=modes[border])
patch = np.arange(1, 26).reshape(5, 5).astype(np.uint8)
print('a 5 × 5 patch padded by 1:\n', cv2.copyMakeBorder(patch, 1, 1, 1, 1, modes[border]))
plt.imshow(blurred); plt.title(f'box {size} × {size}, border = {border}'); plt.axis('off'); plt.show()

**What to notice:** zero padding pulls black into the frame; replicate and reflect do not. `reflect` (OpenCV's default) mirrors across the edge without repeating it.

In [ ]:
# A5 · Output size: out = ⌊(n + 2p − k) / s⌋ + 1
n = 32   # input size · try 7, 224
k = 3    # kernel size · try 5
p = 1    # padding · try 0, 2
s = 2    # stride · try 1, 3

out = (n + 2 * p - k) // s + 1
x = np.zeros((n, n), np.float32)
padded = cv2.copyMakeBorder(x, p, p, p, p, cv2.BORDER_CONSTANT)
real = cv2.filter2D(padded, -1, np.ones((k, k), np.float32))[k // 2:padded.shape[0] - k // 2:s, k // 2:padded.shape[1] - k // 2:s]
print(f'formula: ({n} + 2·{p} − {k}) // {s} + 1 = {out}   ·   counted: {real.shape[0]}')

**What to notice:** 32, k 3, p 1, s 2 gives 16 — the Try it on the slides. Session 4 uses this formula for every layer of a CNN.

## B · Smoothing & sharpening

Averaging neighbours removes noise — and some detail with it. The Gaussian does it more gently; sharpening adds the detail back.

In [ ]:
# B1 · Box vs Gaussian on a noisy photo
noise = 24     # noise level σ · try 10, 50
box = 5        # box size k · try 3, 15
sigma = 1.5    # Gaussian σ · try 0.8, 4

noisy = np.clip(gray + rng.normal(0, noise, gray.shape), 0, 255).astype(np.float32)
by_box = cv2.blur(noisy, (box, box))
by_gauss = cv2.GaussianBlur(noisy, (0, 0), sigma)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
for a, im, t in zip(ax, (noisy, by_box, by_gauss), (f'noise σ = {noise}', f'box {box} × {box}', f'Gaussian σ = {sigma}')):
    a.imshow(im[150:350, 180:380], cmap='gray', vmin=0, vmax=255); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

**What to notice:** both remove grain; the bigger the kernel, the more detail goes with it. The Gaussian keeps edges a little crisper for the same smoothing.

In [ ]:
# B2 · Averaging n pixels cuts the noise by √n (the chart on the slides)
noise = 24      # σ of the noise · try 8, 48
flat = rng.normal(0, noise, (600, 600)).astype(np.float32)   # pure noise, no picture

for k in (1, 3, 5, 7, 9):
    left = cv2.blur(flat, (k, k))[20:-20, 20:-20].std()
    print(f'{k} × {k} box (n = {k * k:2d}): noise left {left:5.2f}   ·   {noise} ÷ √{k * k} = {noise / k:5.2f}')

**What to notice:** the measured noise sits right on σ ÷ √n — the same √N as the photon noise in session 2. Halving the noise costs four times as many pixels.

In [ ]:
# B3 · The Gaussian kernel, and why it is separable
sigma = 1.0   # · try 0.5, 3

g = cv2.getGaussianKernel(int(6 * sigma + 1) | 1, sigma)    # a column, about 6σ + 1 long, odd
kernel2d = g @ g.T                                           # column × row = the 2D kernel
print(f'σ = {sigma}: {kernel2d.shape[0]} × {kernel2d.shape[1]} kernel · centre weight {kernel2d.max():.3f} · sum {kernel2d.sum():.3f}')

img = gray.astype(np.float32)
t0 = time.perf_counter(); full = cv2.filter2D(img, -1, kernel2d); t1 = time.perf_counter()
rows_cols = cv2.sepFilter2D(img, -1, g, g); t2 = time.perf_counter()
print(f'2D: {(t1 - t0) * 1000:.1f} ms · row then column: {(t2 - t1) * 1000:.1f} ms · same result: {np.allclose(full, rows_cols, atol=1e-3)}')
print(f'multiplies per pixel: {kernel2d.size} vs {2 * len(g)}')

**What to notice:** σ = 1 gives a 7 × 7 kernel with centre weight 0.159, as on the slides. Row-then-column gives the same image for 2k multiplies instead of k² (OpenCV may time both alike for small kernels — try σ = 3).

In [ ]:
# B4 · Unsharp masking: sharp = f + α (f − blur(f))
alpha = 1.0   # · try 0, 3
sigma = 2     # how wide the blur is · try 1, 5

f = gray.astype(np.float32)
detail = f - cv2.GaussianBlur(f, (0, 0), sigma)
sharp = np.clip(f + alpha * detail, 0, 255)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
ax[0].imshow(f[100:300, 150:350], cmap='gray', vmin=0, vmax=255); ax[0].set_title('original')
ax[1].imshow(detail[100:300, 150:350], cmap='gray', vmin=-40, vmax=40); ax[1].set_title('detail = f − blur (grey = 0)')
ax[2].imshow(sharp[100:300, 150:350], cmap='gray', vmin=0, vmax=255); ax[2].set_title(f'sharpened, α = {alpha}')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**What to notice:** the detail layer is what the low-pass filter removed. At α = 3 halos appear along edges, and any noise is boosted too.

## C · Edges

An edge is where the numbers jump. A derivative measures the jump; smoothing first keeps noise from winning.

In [ ]:
# C1 · Brightness along one row of the photo
row = 300   # which row · try 100, 450

plt.figure(figsize=(10, 3))
plt.plot(gray[row], color='coral')
plt.xlabel('column'); plt.ylabel('brightness'); plt.title(f'row {row}')
plt.show()
plt.imshow(gray, cmap='gray'); plt.axhline(row, color='coral'); plt.axis('off'); plt.show()

**What to notice:** each jump in the plot is an edge you can find on the photo: a **step** (one side darker), a **ramp** (soft) or a **line** (up and back down).

In [ ]:
# C2 · Derivative = difference of neighbours, and why we smooth first
smooth_first = True   # · try False

img = gray.astype(np.float32)
if smooth_first:
    img = cv2.GaussianBlur(img, (0, 0), 1.0)
gx = cv2.filter2D(img, -1, np.array([[-1, 0, 1]], np.float32))      # right − left
gy = cv2.filter2D(img, -1, np.array([[-1], [0], [1]], np.float32))  # below − above

fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
ax[0].imshow(gx, cmap='gray', vmin=-60, vmax=60); ax[0].set_title('gx (grey = 0)')
ax[1].imshow(gy, cmap='gray', vmin=-60, vmax=60); ax[1].set_title('gy (grey = 0)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**What to notice:** gx lights up upright edges, gy flat ones. Without smoothing the flat areas fill with speckle — a difference doubles the noise.

In [ ]:
# C3 · Sobel at one pixel (the Try it on the slides), then on the whole photo
patch = np.array([[10, 13, 16], [14, 17, 20], [18, 21, 24]], np.float32)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
sobel_y = sobel_x.T                                   # rows count downwards: + means brighter below
gx, gy = (patch * sobel_x).sum(), (patch * sobel_y).sum()
print(f'gx = {gx:.0f} · gy = {gy:.0f} · |∇f| = {np.hypot(gx, gy):.0f} · θ = {np.degrees(np.arctan2(gy, gx)):.1f}°')

threshold = 100   # keep edges stronger than this · try 50, 250
g = gray.astype(np.float32)
mag = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
plt.figure(figsize=(9, 4.5))
plt.subplot(1, 2, 1); plt.imshow(mag, cmap='gray_r', vmax=400); plt.title('|∇f| (dark = strong)'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(mag > threshold, cmap='gray_r'); plt.title(f'|∇f| > {threshold}'); plt.axis('off')
plt.tight_layout(); plt.show()

**What to notice:** 24, 32, 40 and 53° — the 3-4-5 triangle from the slides. On the photo, one threshold gives thick edges that break up: that is what Canny fixes.

In [ ]:
# C4 · Gradient direction as colour: hue = direction, brightness = strength
g = cv2.GaussianBlur(gray.astype(np.float32), (0, 0), 1.0)
gx, gy = cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)
strength = np.clip(np.hypot(gx, gy) / 300, 0, 1)
direction = (np.degrees(np.arctan2(gy, gx)) % 360) / 2          # OpenCV hue: 0–179

hsv = np.dstack([direction, np.full_like(direction, 255), strength * 255]).astype(np.uint8)
plt.figure(figsize=(5, 5)); plt.imshow(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)); plt.axis('off')
plt.title('colour = which way is brighter'); plt.show()

**What to notice:** the two sides of a thin line get opposite colours — the gradient points towards brighter, and the edge runs across it at 90°.

In [ ]:
# C5 · Canny: thin, connected edges from two thresholds
low, high = 50, 150   # weak and strong thresholds
sigma = 1.4           # smoothing before the gradient · try 0.5, 3

blurred = cv2.GaussianBlur(gray, (0, 0), sigma)
edges = cv2.Canny(blurred, low, high)
plt.figure(figsize=(5, 5)); plt.imshow(255 - edges, cmap='gray'); plt.axis('off')
plt.title(f'Canny · σ {sigma} · thresholds {low}/{high}'); plt.show()
print(f'{(edges > 0).mean():.1%} of the pixels are edges')

**What to notice:** more smoothing → fewer, longer edges; less → texture and noise come back. The edges are one pixel thin — non-maximum suppression at work.

## D · Frequency & sampling

Every image is a sum of waves. Smoothing keeps the slow ones; shrinking an image can turn fast ones into false, slow ones.

In [ ]:
# D1 · A square wave from sine waves
n_waves = 3   # how many odd sines (1, 3, 5, …) · try 1, 30

x = np.linspace(0, 4 * np.pi, 1000)
total = np.zeros_like(x)
for k in range(1, n_waves + 1):
    m = 2 * k - 1
    total += 4 / np.pi * np.sin(m * x) / m
plt.figure(figsize=(10, 3))
plt.plot(x, np.sign(np.sin(x)), color='#cbd5e1', lw=3, label='square wave')
plt.plot(x, total, color='teal', label=f'sum of {n_waves} sines'); plt.legend(); plt.show()

**What to notice:** the sharp step needs the fast waves. Images work the same way in 2D: edges and noise live in the high frequencies.

In [ ]:
# D2 · The Fourier transform: keep the slow waves (low-pass) or the fast ones (high-pass)
radius = 20      # · try 5, 60
keep = 'low'     # 'low' or 'high' · try 'high'

F = np.fft.fftshift(np.fft.fft2(gray.astype(float)))     # centre = slowest waves
h, w = gray.shape
yy, xx = np.mgrid[-h // 2:h // 2, -w // 2:w // 2]
mask = np.hypot(xx, yy) <= radius
if keep == 'high':
    mask = ~mask
rebuilt = np.real(np.fft.ifft2(np.fft.ifftshift(F * mask)))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.8))
ax[0].imshow(np.log1p(np.abs(F)) * mask, cmap='gray'); ax[0].set_title('Fourier magnitude (kept part)')
ax[1].imshow(rebuilt, cmap='gray'); ax[1].set_title(f'rebuilt from the {keep} frequencies')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**What to notice:** low-pass is a blur, high-pass leaves the edges. Most of the strength sits near the centre — real images are mostly slow change.

In [ ]:
# D3 · Nyquist: at least two samples per cycle (the fence on the slides)
period = 6   # the pattern repeats every 6 pixels
step = 4     # keep every step-th pixel · try 2, 3, 8

x = np.arange(240)
stripes = np.cos(2 * np.pi * x / period)
kept = stripes[::step]
spectrum = np.abs(np.fft.rfft(kept)); spectrum[0] = 0
seen = len(kept) * step / spectrum.argmax()
print(f'{period / step:.2f} samples per cycle → the kept pixels repeat every {seen:.0f} px'
      + ('  ✓ the true pattern' if period / step > 2 else '  ← exactly 2: the limit, not safe' if period / step == 2 else '  ← aliasing: a false pattern'))

**What to notice:** step 4 gives 1.5 samples per cycle and a false 12 px pattern — the Try it on the slides. Step 3 is exactly 2 per cycle: here the samples happen to land on the peaks, but shift the stripes by a quarter cycle and they would all read 0 — the limit is not safe. Step 2 (3 per cycle) keeps the true 6 px.

In [ ]:
# D4 · Shrink by skipping pixels vs blur first, then skip
step = 4            # shrink factor · try 2, 6
picture = 'rings'   # 'rings' (test pattern) or 'brick' (a photo) · try 'brick'

if picture == 'rings':
    yy, xx = np.mgrid[0:1024, 0:1024].astype(np.float32)
    big = 127.5 + 127.5 * np.cos(np.pi * ((xx - 512) ** 2 + (yy - 512) ** 2) / 2048)   # finer rings further out
else:
    big = cv2.resize(data.brick(), (1024, 1024)).astype(np.float32)
skipped = big[::step, ::step]
smoothed = cv2.GaussianBlur(big, (0, 0), 0.6 * step)[::step, ::step]

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(skipped, cmap='gray', interpolation='nearest'); ax[0].set_title(f'keep every {step}th pixel')
ax[1].imshow(smoothed, cmap='gray', interpolation='nearest'); ax[1].set_title(f'Gaussian σ = {0.6 * step:.1f} first')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**What to notice:** skipping makes rings that are not in the picture (aliasing); blurring first turns detail that is too fine into flat grey instead.

In [ ]:
# D5 · Gaussian pyramid: blur, then halve — again and again
levels = 5   # · try 3, 7

pyramid = [photo]
for _ in range(levels - 1):
    pyramid.append(cv2.pyrDown(pyramid[-1]))
fig, ax = plt.subplots(1, levels, figsize=(12, 3), gridspec_kw={'width_ratios': [p.shape[1] for p in pyramid]})
for a, im in zip(ax, pyramid):
    a.imshow(im); a.set_title(f'{im.shape[1]} × {im.shape[0]}'); a.axis('off')
plt.show()
total = sum(im.shape[0] * im.shape[1] for im in pyramid)
print(f'all levels together: {total / (photo.shape[0] * photo.shape[1]):.3f} × the original pixels')

**What to notice:** each level has ¼ of the pixels, so the whole pyramid costs only about ⅓ more than the original. Detectors run on every level to find objects of every size (sessions 6 and 8).

---
Back to the slides: [session 3 deck](https://classes.incortx.com/ComputerVision/session-03/slides.html) · exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s3).